In [ ]:

import logging
from validate_credentials import validate_credentials
from send_link import send_link
from remote_desktop import transmit
from daemon_client import send_to_daemon, ping, load_fernet

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Minimum required fields for a transmittable client package
REQUIRED_CLIENT_PKG_FIELDS = {"key", "endpoint", "transport"}



In [ ]:
def build_client_pkg(process: dict) -> dict:
    """
    Extract and validate the minimum transmit signature from a process dict.
    Raises KeyError if required fields are missing.
    """
    pkg = process.get("client_pkg", {})
    missing = REQUIRED_CLIENT_PKG_FIELDS - pkg.keys()
    if missing:
        raise KeyError(f"client_pkg missing required fields: {missing}")
    return pkg


In [ ]:
def send_credential(process: dict, origin: dict) -> bool:
    """
    Validate credentials, establish landing zone, transmit credentials.

    Transport options in client_pkg:
        "webapp"  — HTTP POST via Flask server (remote_desktop.py)
        "ssh"     — remote bash script via OpenSSH
        "daemon"  — direct TCP socket to credential daemon (ssh_daemon.py port 9922)

    Args:
        process: must contain 'client_pkg' with keys: endpoint, key, transport
        origin:  reference credential dict for validation

    Returns:
        True on successful transmission, False otherwise
    """
    try:
        client_pkg = build_client_pkg(process)
    except KeyError as e:
        log.error(f"Malformed process package: {e}")
        return False

    if not validate_credentials(origin, client_pkg):
        log.warning("Credential validation failed — aborting transmission")
        return False

    transport = client_pkg.get("transport")

    try:
        # ── Webapp transport ──────────────────────────────────────────────
        if transport == "webapp":
            log.info("Establishing webapp landing zone...")
            send_link(process)
            transmit(client_pkg)

        # ── SSH transport ─────────────────────────────────────────────────
        elif transport == "ssh":
            from ssh import ssh
            username = client_pkg.get("username")
            ip       = client_pkg.get("endpoint")
            if not username or not ip:
                raise ValueError("SSH transport requires 'username' and 'endpoint' in client_pkg")
            log.info(f"Initiating SSH transmission to {username}@{ip}...")
            ssh(username, ip)

        # ── Daemon transport ──────────────────────────────────────────────
        elif transport == "daemon":
            host      = client_pkg.get("endpoint")
            port      = int(client_pkg.get("daemon_port", 9922))
            key_path  = client_pkg.get("fernet_key_path", "fernet.key")

            if not host:
                raise ValueError("Daemon transport requires 'endpoint' in client_pkg")

            fernet = load_fernet(key_path)

            # Confirm daemon is alive before attempting delivery
            if not ping(host, port):
                raise ConnectionRefusedError(
                    f"Daemon not responding at {host}:{port} — "
                    "check that credential-daemon.service is running on the target machine"
                )

            result = send_to_daemon(
                credential=client_pkg,
                host=host,
                fernet=fernet,
                port=port,
            )
            log.info(f"Stored in daemon ledger — block #{result['block_index']}")

        else:
            raise ValueError(
                f"Unknown transport: '{transport}' — expected 'webapp', 'ssh', or 'daemon'"
            )

        log.info("Transmission complete.")
        return True

    except Exception as e:
        log.error(f"Transmission failed: {e}")
        return False